In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

PROCESSED_DIR = Path('../../data/processed')
OUTPUT_DIR = Path('outputs/sephora_segmentation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

K_RANGE = range(2, 11)

In [4]:
df = pd.read_csv(PROCESSED_DIR / 'sephora_segmentation.csv')
print(f'{len(df):,} products loaded')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/sephora_segmentation.csv'

In [ ]:
core_features = [
    'avg_rating', 'rating_std',
    'pct_5_star', 'pct_4_star', 'pct_3_star', 'pct_2_star', 'pct_1_star',
    'avg_sentiment', 'pct_positive', 'pct_negative',
    'reviews_per_month', 'review_count_normalized',
    'rating_dispersion_over_time', 'top_topic_prevalence',
    'dominant_topic_mode', 'price', 'price_vs_category_median',
    'avg_helpfulness',
]

skin_tone_cols = [c for c in df.columns if c.startswith('pct_skin_tone_')]
skin_type_cols = [c for c in df.columns if c.startswith('pct_skin_type_')]
coverage_cols = [c for c in df.columns if c in ('pct_has_skin_tone', 'pct_has_skin_type')]

feature_names = [f for f in core_features + skin_tone_cols + skin_type_cols + coverage_cols if f in df.columns]
print(f'Selected {len(feature_names)} features')

X = df[feature_names].apply(pd.to_numeric, errors='coerce').fillna(0)
weights = np.clip(pd.to_numeric(df['review_count_normalized'], errors='coerce').fillna(0).values, 0.1, 1.0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Feature matrix: {X_scaled.shape}')

In [ ]:
def plot_silhouette_detail(X_scaled, labels, title):
    fig, ax = plt.subplots(figsize=(10, 8))
    sil_vals = silhouette_samples(X_scaled, labels)
    n_clusters = len(set(labels))
    y_lower = 10

    for i in range(n_clusters):
        cluster_sil = np.sort(sil_vals[labels == i])
        size = len(cluster_sil)
        y_upper = y_lower + size
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil, alpha=0.7)
        ax.text(-0.05, y_lower + 0.5 * size, str(i), fontsize=10, fontweight='bold')
        y_lower = y_upper + 10

    avg_sil = sil_vals.mean()
    ax.axvline(avg_sil, color='red', linestyle='--', label=f'Avg: {avg_sil:.3f}')
    ax.set_xlabel('Silhouette Coefficient')
    ax.set_ylabel('Cluster')
    ax.set_title(title)
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.show()


profile_features = [
    'avg_rating', 'avg_sentiment', 'price', 'reviews_per_month',
    'pct_5_star', 'pct_1_star', 'pct_positive', 'pct_negative',
    'rating_std', 'price_vs_category_median', 'review_count_normalized',
    'top_topic_prevalence', 'rating_dispersion_over_time',
]
profile_features = [f for f in profile_features if f in df.columns]


def profile_clusters(df, col):
    profiles = df.groupby(col)[profile_features].mean()
    profiles_norm = (profiles - profiles.min()) / (profiles.max() - profiles.min() + 1e-9)

    fig, ax = plt.subplots(figsize=(14, max(6, len(profiles) * 1.5)))
    sns.heatmap(profiles_norm.T, annot=profiles.T.round(3), fmt='', cmap='YlOrRd',
                xticklabels=[f'Cluster {i}' for i in profiles.index],
                yticklabels=profiles.columns, ax=ax, linewidths=0.5)
    ax.set_title(f'Cluster Profiles — {col}', fontsize=14)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'profile_{col}.png', dpi=150)
    plt.show()

    for cluster_id in sorted(df[col].unique()):
        c = df[df[col] == cluster_id]
        print(f'\n  Cluster {cluster_id} ({len(c):,} products, {len(c)/len(df)*100:.1f}%)')
        print(f'  Top categories: {c["category"].value_counts().head(5).to_dict()}')
        print(f'  Top brands: {c["brand"].value_counts().head(5).to_dict()}')
        print(f'  Price: ${c["price"].min():.0f}-${c["price"].max():.0f} (median ${c["price"].median():.0f})')
        print(f'  Avg rating: {c["avg_rating"].mean():.2f}  |  Avg sentiment: {c["avg_sentiment"].mean():.3f}')
        print(f'  Reviews/month: {c["reviews_per_month"].mean():.1f}  |  Rating std: {c["rating_std"].mean():.3f}')

# Model 1: K-Means Unweighted

In [ ]:
km_uw_inertias, km_uw_silhouettes, km_uw_models = [], [], {}

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    km.fit(X_scaled)
    sil = silhouette_score(X_scaled, km.labels_)
    km_uw_inertias.append(km.inertia_)
    km_uw_silhouettes.append(sil)
    km_uw_models[k] = km
    print(f'  k={k}: WCSS={km.inertia_:,.0f}  silhouette={sil:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.plot(list(K_RANGE), km_uw_inertias, 'o-', linewidth=2, color='steelblue')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Within-Cluster Sum of Squares')
ax1.set_title('K-Means Unweighted — Elbow Plot')
ax1.grid(True, alpha=0.3)

ax2.plot(list(K_RANGE), km_uw_silhouettes, 'o-', linewidth=2, color='darkorange')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('K-Means Unweighted — Silhouette Scores')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'km_uw_elbow_silhouette.png', dpi=150)
plt.show()

In [ ]:
KM_UW_K = 4

In [ ]:
df['kmeans_unweighted'] = km_uw_models[KM_UW_K].labels_
sil = silhouette_score(X_scaled, df['kmeans_unweighted'].values)
print(f'K-Means Unweighted (k={KM_UW_K}): silhouette={sil:.4f}')

plot_silhouette_detail(X_scaled, df['kmeans_unweighted'].values,
                       f'K-Means Unweighted (k={KM_UW_K})')

profile_clusters(df, 'kmeans_unweighted')